In [7]:
!pip install -q requests beautifulsoup4 nltk
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter
import re

def setup_nltk():
    try:
        nltk.data.find('sentiment/vader_lexicon')
    except LookupError:
        nltk.download('vader_lexicon')

    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')


class NewsAnalyzer:

    def __init__(self):
        setup_nltk()
        self.analyzer = SentimentIntensityAnalyzer()

    def is_valid_url(self, url):
        pattern = r"^https?://[^\s]+$"
        return bool(re.match(pattern, url))

    def fetch_article(self, url):
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')

            article_text = " ".join([p.get_text() for p in paragraphs])

            if not article_text.strip():
                print("⚠️ No readable content found.")
                return None

            return article_text

        except requests.exceptions.RequestException as e:
            print(f"❌ Network error: {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            return None

    def check_sentiment(self, text):
        scores = self.analyzer.polarity_scores(text)

        if scores['compound'] >= 0.05:
            label = "Positive 😊"
        elif scores['compound'] <= -0.05:
            label = "Negative 😠"
        else:
            label = "Neutral 😐"

        return label, scores

    def make_summary(self, text, num_sentences=3):
        sentences = sent_tokenize(text)

        if len(sentences) <= num_sentences:
            return text

        words = word_tokenize(text.lower())
        word_freq = Counter(words)

        sentence_scores = {}

        for sentence in sentences:
            for word in word_tokenize(sentence.lower()):
                if word in word_freq:
                    if sentence not in sentence_scores:
                        sentence_scores[sentence] = word_freq[word]
                    else:
                        sentence_scores[sentence] += word_freq[word]

        sorted_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)
        selected_sentences = sorted_sentences[:num_sentences]
        final_summary = [s for s in sentences if s in selected_sentences]

        return " ".join(final_summary)


def main():
    analyzer = NewsAnalyzer()

    print("\n===== 📰 Zain's News Sentiment Analyzer =====\n")

    user_url = input("🔗 Enter article URL: ").strip()

    if not analyzer.is_valid_url(user_url):
        print("❌ Error: Invalid URL format.")
        return

    try:
        input_val = input("📝 Number of summary sentences (default 3): ").strip()
        num_sentences = int(input_val) if input_val else 3
        if num_sentences <= 0:
            num_sentences = 3
    except ValueError:
        num_sentences = 3

    article = analyzer.fetch_article(user_url)

    if article:
        sentiment, scores = analyzer.check_sentiment(article)
        summary = analyzer.make_summary(article, num_sentences)

        print("\n===== 📊 Analysis Result =====")
        print(f"\nSentiment: {sentiment}")

        print("\nDetailed Scores:")
        for key, value in scores.items():
            print(f"{key}: {value}")

        print("\n📝 Summary:\n")
        print(summary)

    else:
        print("❌ Error: Unable to fetch article. Try another URL.")


if __name__ == "__main__":
    main()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!



===== 📰 Zain's News Sentiment Analyzer =====

🔗 Enter article URL: https://www.bbc.com/news/live/cqxdg17yr2wt
📝 Number of summary sentences (default 3): 2

===== 📊 Analysis Result =====

Sentiment: Positive 😊

Detailed Scores:
neg: 0.06
neu: 0.83
pos: 0.11
compound: 0.9995

📝 Summary:

The Student News Network (SNN), linked to the IRGC's
paramilitary Basij Student Organisation, called for âclarificationâ by Iranian
authorities so that the âinterpretationâ of Donald Trumpâs âvictoryâ in
relation to this matter will be âchallenged and dismantled.â The Fars news agency - also affiliated with the IRGC - has
also called on the authorities to "clarify" the matter, saying that
"short, pithy statements" on X are "not suitable" for persuading
"domestic public opinion" of Iranians. He has posted on X saying: âWe are currently verifying the
recent announcement related to the reopening of the Strait of Hormuz, in terms
of its compliance with freedom of navigation for all merch